# Task 3 - Optimizers

**Requirements:**
 - numpy (https://numpy.org/)
 - matplotlib (https://matplotlib.org/)

 Last task. Let's continue with our framework. We use all of the previous implemented classes and add new - **Optimizers**...


In [1]:
# Import
import numpy as np
from utils import Module
import plotly.express as px

In [2]:
#------------------------------------------------------------------------------
#   Linear layer (Dense, Fully connected, Single Layer Perceptron)
#------------------------------------------------------------------------------
class Linear(Module):
    def __init__(self, in_features, out_features):
        super(Linear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = np.random.randn(out_features, in_features)
        self.dW = np.zeros_like(self.W) # Watch-out for the shape - it has to be same as W
        self.b = np.zeros((out_features, 1))
        self.db = np.zeros_like(self.b) # Watch-out for the shape - it has to be same as b

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_inputs = input
        self.m = self.fw_inputs.shape[1]
        net = np.matmul(self.W, input) + self.b
        return net

    def backward(self, dz: np.ndarray) -> np.ndarray:
        self.dW = (1.0/self.m) * np.matmul(dz, self.fw_inputs.T)
        self.db = (1.0/self.m) * np.sum(dz, axis=1, keepdims=True)
        return np.matmul(self.W.T, dz)

    def get_optimizer_context(self):
        return [[self.W, self.dW], [self.b, self.db]]

    def set_optimizer_context(self, params):
        self.W, self.b = params

#------------------------------------------------------------------------------
#   SigmoidActivationFunction class
#------------------------------------------------------------------------------
class Sigmoid(Module):
    def __init__(self):
        super(Sigmoid, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return 1.0 / (1.0 + np.exp(-input))

    def backward(self, da) -> np.ndarray:
        a = self(self.fw_input)
        return np.multiply(da, np.multiply(a, 1 - a))

#------------------------------------------------------------------------------
#   HyperbolicTangentActivationFunction class
#------------------------------------------------------------------------------
class Tanh(Module):
    def __init__(self):
        super(Tanh, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return (np.exp(2 * input) - 1) / (np.exp(2 * input) + 1)

    def backward(self, da) -> np.ndarray:
        a = self(self.fw_input)
        return np.multiply(da, 1 - np.square(a))

#------------------------------------------------------------------------------
#   Model class
#------------------------------------------------------------------------------
class Model(Module):
    def __init__(self):
        super(Model, self).__init__()

    def forward(self, input) -> np.ndarray:
        for name, module in self.modules.items():
            input = module(input)
        return input

    def backward(self, dz: np.ndarray):
        for name, module in reversed(self.modules.items()):
            dz = module.backward(dz)


## Loss Functions

As in standard deep learning frameworks, calling Loss function can return either **cost** or  **loss**  based on parameter **reduce**.

In [3]:
#------------------------------------------------------------------------------
#   MeanSquareErrorLossFunction class
#------------------------------------------------------------------------------
class MSELoss(Module):
    def __init__(self, reduce="mean"):
        super(MSELoss, self).__init__()
        if reduce == "mean":
            self.reduce_fn = np.mean
        elif reduce == "sum":
            self.reduce_fn = np.sum
        elif reduce is None:
            # return identity (do nothing)
            self.reduce_fn = lambda x : x
        else:
            raise AttributeError

    def forward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        return self.reduce_fn(np.power(target - input, 2))

    def backward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        return -2 * (target - input)


#------------------------------------------------------------------------------
#   BinaryCrossEntropyLossFunction class
#------------------------------------------------------------------------------
class BCELoss(Module):
    def __init__(self, reduce="mean"):
        super(BCELoss, self).__init__()
        if reduce == "mean":
            self.reduce_fn = np.mean
        elif reduce == "sum":
            self.reduce_fn = np.sum
        elif reduce is None:
            # return identity (do nothing)
            self.reduce_fn = lambda x : x
        else:
            raise AttributeError

    def forward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        return self.reduce_fn(-(target * np.log(input) + np.multiply((1 - target), np.log(1 - input))))

    def backward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        return -np.divide(target, input) + np.divide(1 - target, 1 - input)

## Optimizers

Each optimizer gets as input a **model** and loads each layer's parameters for optimizer context **`layer.get_optimizer_context()`**. Other attributes are based on the optimizer definition. The modified parameters are put back to the model's layer by `layer.set_optimizer_context([W,b])`. Remember that optimizers may require to store some context for the next steps of optimization for each layer and each parameter accordingly.

Your task is to implement:
 - SGD with momentum
 - RMSProp: http://www.cs.toronto.edu/~hinton/coursera/lecture6/lec6.pdf
 - Adam: https://arxiv.org/pdf/1412.6980.pdf

All algorithms are in [https://www.deeplearningbook.org/contents/optimization.html](https://www.deeplearningbook.org/contents/optimization.html)


In [4]:
#------------------------------------------------------------------------------
#   AbstractOptimizer class
#------------------------------------------------------------------------------
class Optimizer:
    def __init__(self):
        pass

    def step(self, model):
        raise NotImplemented

#------------------------------------------------------------------------------
#   StochasticGradientDescentOptimizer class
#------------------------------------------------------------------------------
class SGD(Optimizer):
    def __init__(self, lr:float):
        super(SGD, self).__init__()
        self.lr = lr

    def step(self, model):
        for name, layer in model.modules.items():
            if hasattr(layer, 'get_optimizer_context'):
                params = layer.get_optimizer_context()
                if params is not None:
                    [[W, dW],[b,db]] = params
                    # >>>> start here

                    # Update parameters
                    W = W - self.lr * dW
                    b = b - self.lr * db
                    
                    # <<<< end here
                    layer.set_optimizer_context([W,b])


In [5]:
#------------------------------------------------------------------------------
#   SGDMomentumOptimizer class
#------------------------------------------------------------------------------
class SGDMomentum(Optimizer):
    def __init__(self, lr, beta):
        super(SGDMomentum, self).__init__()
        self.context = {}
        # >>>> start_solution
        self.lr = lr
        self.beta = beta
        # <<<< end_solution

    def step(self, model):
        for name, layer in model.modules.items():
            if hasattr(layer, 'get_optimizer_context'):
                params = layer.get_optimizer_context()
                if params is not None:
                    [[W, dW],[b,db]] = params

                    # >>>> start_solution

                    if (not name in self.context.keys()):
                        self.context[name] = {
                            'VdW': np.zeros_like(W),
                            'Vdb': np.zeros_like(b)
                        }

                    self.context[name]['VdW'] = (1 - self.beta) * dW + self.beta * self.context[name]['VdW']
                    self.context[name]['Vdb'] = (1 - self.beta) * db + self.beta * self.context[name]['Vdb']

                    W = W - self.lr * self.context[name]['VdW']
                    b = b - self.lr * self.context[name]['Vdb']
                        
                    # <<<< end_solution
                    layer.set_optimizer_context([W,b])

In [6]:
#------------------------------------------------------------------------------
#   RMSpropOptimizer class
#------------------------------------------------------------------------------
class RMSprop(Optimizer):
    def __init__(self, lr, beta):
        super(RMSprop, self).__init__()
        self.context = {}
        # >>>> start_solution
        self.lr = lr
        self.beta = beta
        # <<<< end_solution

    def step(self, model):
        for name, layer in model.modules.items():
            if hasattr(layer, 'get_optimizer_context'):
                params = layer.get_optimizer_context()
                if params is not None:
                    [[W, dW], [b, db]] = params

                    # >>>> start_solution

                    if (not name in self.context.keys()):
                        self.context[name] = {
                            'SdW': np.zeros_like(W),
                            'Sdb': np.zeros_like(b)
                        }

                    self.context[name]['SdW'] = (1 - self.beta) * dW ** 2 + self.beta * self.context[name]['SdW']
                    self.context[name]['Sdb'] = (1 - self.beta) * db ** 2 + self.beta * self.context[name]['Sdb']

                    W = W - self.lr * dW / np.sqrt(self.context[name]['SdW'] + 1e-8)
                    b = b - self.lr * db / np.sqrt(self.context[name]['Sdb'] + 1e-8)

                    # <<<< end_solution
                    layer.set_optimizer_context([W, b])

In [7]:
#------------------------------------------------------------------------------
#   AdamOptimizer class
#------------------------------------------------------------------------------
class Adam(Optimizer):
    def __init__(self, lr, beta1, beta2):
        super(Adam, self).__init__()
        self.context = {}
        # >>>> start_solution
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.t = 0
        # <<<< end_solution

    def step(self, model):
        # >>>>>> Probably add something here ;)
        self.t += 1
        # <<<<<< until here
        for name, layer in model.modules.items():
            if hasattr(layer, 'get_optimizer_context'):
                params = layer.get_optimizer_context()
                if params is not None:
                    [[W, dW], [b, db]] = params

                    # >>>> start_solution

                    if (not name in self.context.keys()):
                        self.context[name] = {
                            'VdW': np.zeros_like(W),
                            'SdW': np.zeros_like(W),
                            'Vdb': np.zeros_like(b),
                            'Sdb': np.zeros_like(b)
                        }

                    self.context[name]['VdW'] = (1 - self.beta1) * dW + self.beta1 * self.context[name]['VdW']
                    self.context[name]['SdW'] = (1 - self.beta2) * dW ** 2 + self.beta2 * self.context[name]['SdW']
                    self.context[name]['Vdb'] = (1 - self.beta1) * db + self.beta1 * self.context[name]['Vdb']
                    self.context[name]['Sdb'] = (1 - self.beta2) * db ** 2 + self.beta2 * self.context[name]['Sdb']

                    V_hat_dW = self.context[name]['VdW'] / (1 - self.beta1 ** self.t)
                    S_hat_dW = self.context[name]['SdW'] / (1 - self.beta2 ** self.t)
                    V_hat_db = self.context[name]['Vdb'] / (1 - self.beta1 ** self.t)
                    S_hat_db = self.context[name]['Sdb'] / (1 - self.beta2 ** self.t)

                    W = W - self.lr * V_hat_dW / (np.sqrt(S_hat_dW) + 1e-8)
                    b = b - self.lr * V_hat_db / (np.sqrt(S_hat_db) + 1e-8)

                    # <<<< end_solution
                    layer.set_optimizer_context([W, b])

## Main Processing Cell

Watch out for the shape of mini-batch (num_features, batch_size)

 1. Initialize dataset (`dataset_Flower`).
 2. Declare a simple model.
 3. Initialize optimizer.
 4. Make mini-batches.
 5. Perform forward pass through the network.
 6. Compute loss.
 7. Backward prop loss.
 8. Track loss.
 9. Backward pass MLP.
 10. Use optimizer to modify model parameters.
 11. Repeat for $N$ epochs

In [8]:
from utils import gradient_check
from dataset import dataset_Flower, MakeBatches

In [9]:
#
#   This is our model - you don't need to modify it
#
def create_model():
    mlp = Model()
    mlp.add_module(Linear(2, 3), 'Dense_1')
    mlp.add_module(Tanh(), 'Tanh_1')
    mlp.add_module(Linear(3, 4), 'Dense_2')
    mlp.add_module(Tanh(), 'Tanh_2')
    mlp.add_module(Linear(4, 5), 'Dense_3')
    mlp.add_module(Tanh(), 'Tanh_3')
    mlp.add_module(Linear(5, 1), 'Dense_4_out')
    mlp.add_module(Sigmoid(), 'Sigmoid')
    return mlp


In [10]:
#
#   This performs a single experiment using the provided model, optimizer, dataset, and loss function
#   You don't need to modify it
#
def fit(model, optimizer, dataset, criterion, num_epochs=1000):
    """ Executes the training loop and returns array of training losses for each epoch """
    result_losses = []
    for epoch in range(num_epochs):

        epoch_loss = []
        for mini_batch_x, mini_batch_y in dataset:
            y_hat = model.forward(mini_batch_x)
            loss = criterion(y_hat, mini_batch_y)
            model.backward(criterion.backward(y_hat, mini_batch_y))
            optimizer.step(model)
            epoch_loss.append( np.mean(loss) )

        result_losses.append( np.mean(epoch_loss) )

    return result_losses


In [11]:
dataset = MakeBatches(dataset_Flower(m=512, noise=0.3), 32, True)

optimizers = {
    "SGD": SGD(lr=0.001),
    "SGD Momentum": SGDMomentum(lr=0.001, beta=0.9),
    "RMSProp": RMSprop(lr=0.001, beta=0.9),
    "Adam": Adam(lr=0.001, beta1=0.95, beta2=0.999)
}

results = { }
for name, opt in optimizers.items():
    print(f"Training using {name} ...")
    results[name] = fit(
        model=create_model(),       # Always start with a new randomly-initialized model
        optimizer=opt,
        dataset=dataset,
        criterion=BCELoss(reduce="mean"),
        num_epochs=1000
    )

fig = px.line(results)
fig.show()

Training using SGD ...


Training using SGD Momentum ...


Training using RMSProp ...


Training using Adam ...
